In [1]:
import os, sys
import pandas as pd
import numpy as np
import sklearn
from sklearn.metrics import mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt
from fastparquet import write
import scikit_posthocs as sp
from scipy import stats

cwd = os.getcwd()
root_path = cwd
while '.venv' not in os.listdir(root_path):
    root_path =  os.path.dirname(root_path)
results_path = os.path.join(root_path,'results')
metrics_path = os.path.join(results_path,'metrics')
forecast_path = os.path.join(results_path,'forecast')
display(forecast_path)

'C:\\Users\\Utilisateur\\Desktop\\doctorat\\2. Recherche\\Article 1\\3. Python\\PhD_article_1\\results\\forecast'

In [27]:
def add_stars(diff, p):
    diff_str = f"{diff:.1e}"  # écriture scientifique, 2 décimales
    if p < 0.001:
        stars = '***'
    elif p < 0.01:
        stars = '**'
    elif p < 0.05:
        stars = '*'
    else:
        stars = ''
    return f"{diff_str}{stars}"

In [40]:
df = pd.read_parquet(os.path.join(metrics_path, "metrics_row.parquet.gzip"))
d = ['OHLCV','FOB']
m = ['CNN','LSTM']
df['data'] = df['data_mod'].apply(lambda x: '_'.join([i for i in x.split('_') if i in d]))
df['model'] = df['data_mod'].apply(lambda x: '_'.join([i for i in x.split('_') if i in m]))
display(df)
for m in ['mse','mae','accuracy']:
    desc = df.groupby(['data','model'])[m].describe()
    pivot = desc.copy()
    pivot['skew'] = df.groupby(['data','model'])[m].skew()
    pivot['kurt'] = df.groupby(['data','model'])[m].apply(pd.DataFrame.kurt)
    pivot = pivot[['mean', '50%', 'skew', 'kurt']]
    pivot.columns = ['M', 'Mdn', 'Skew', 'Kurt']
    """if m == 'accuracy':
        pivot[['M', 'Mdn']] = pivot[['M', 'Mdn']].applymap(lambda x: '{:.2f}'.format(x))
    else:
        pivot[['M', 'Mdn']] = pivot[['M', 'Mdn']].applymap(lambda x: '{:.1e}'.format(x))
    pivot[['Skew', 'Kurt']] = pivot[['Skew', 'Kurt']].applymap(lambda x: '{:.2f}'.format(x))"""
    if m == 'accuracy':
        display(pivot.sort_values('Mdn', ascending=False))
    else: display(pivot.sort_values('Mdn', ascending=True))
        
    pivot = pivot.unstack(0).swaplevel(axis=1).sort_index(axis=1, level=0, sort_remaining=False)#.round(2)
    display(pivot)
    #pivot.to_csv(f"desc_stat_{m}.csv")

,mse,isin,data_mod,id_forecast,accuracy,mae,data,model
0,0.000027,FR0000130577,FOB_LSTM,0,0.56,0.003765,FOB,LSTM
1,0.000028,FR0000130577,FOB_LSTM,1,0.54,0.003831,FOB,LSTM
2,0.000027,FR0000130577,FOB_LSTM,2,0.57,0.003886,FOB,LSTM
3,0.000027,FR0000130577,FOB_LSTM,3,0.57,0.003947,FOB,LSTM
4,0.000027,FR0000130577,FOB_LSTM,4,0.59,0.004146,FOB,LSTM
...,...,...,...,...,...,...,...,...
116174,0.000055,FR0000120172,FOB_CNN_LSTM,393,0.01,0.006758,FOB,CNN_LSTM
116175,0.000031,FR0000120172,FOB_CNN_LSTM,394,0.07,0.004887,FOB,CNN_LSTM
116176,0.000021,FR0000120172,FOB_CNN_LSTM,395,0.17,0.003853,FOB,CNN_LSTM
116177,0.000030,FR0000120172,FOB_CNN_LSTM,396,0.10,0.004649,FOB,CNN_LSTM


M       Mdn       Skew        Kurt
data      model                                              
OHLCV_FOB CNN_LSTM  0.000441  0.000061   8.728737   88.930273
FOB       CNN_LSTM  0.002297  0.000066  11.766600  147.619031
          LSTM      0.001738  0.000072  14.558502  233.083286
OHLCV     CNN_LSTM  0.001484  0.000076  15.325526  253.931667
OHLCV_FOB LSTM      0.002080  0.000080  12.397251  177.164021
          CNN       0.002109  0.000147  14.505459  228.817024
FOB       CNN       0.002117  0.000154  14.756363  237.288173
OHLCV     LSTM      0.001543  0.000155  16.042337  276.305306
          CNN       0.008853  0.002713   6.977890   68.603734

data           FOB                                      OHLCV            \
                 M       Mdn       Skew        Kurt         M       Mdn   
model                                                                     
CNN       0.002117  0.000154  14.756363  237.288173  0.008853  0.002713   
CNN_LSTM  0.002297  0.000066  11.766600  147.619031  0.001484  0.000076   
LSTM      0.001738  0.000072  14.558502  233.083286  0.001543  0.000155   

data                            OHLCV_FOB                                   
               Skew        Kurt         M       Mdn       Skew        Kurt  
model                                                                       
CNN        6.977890   68.603734  0.002109  0.000147  14.505459  228.817024  
CNN_LSTM  15.325526  253.931667  0.000441  0.000061   8.728737   88.930273  
LSTM      16.042337  276.305306  0.002080  0.000080  12.397251  177.164021

M       Mdn       Skew        Kurt
data      model                                              
OHLCV_FOB CNN_LSTM  0.010425  0.006444   5.113143   34.866629
FOB       CNN_LSTM  0.014212  0.006736   9.383533  103.807379
          LSTM      0.014201  0.007074   9.195510  109.425004
OHLCV     CNN_LSTM  0.014125  0.007375  10.118810  134.723838
OHLCV_FOB LSTM      0.015216  0.007383   8.023178   79.760156
          CNN       0.018430  0.010229   9.246033  115.284693
FOB       CNN       0.019089  0.010420   8.841193  106.753009
OHLCV     LSTM      0.016194  0.010481  11.062979  159.354731
          CNN       0.056249  0.041999   3.115163   14.119886

data           FOB                                     OHLCV            \
                 M       Mdn      Skew        Kurt         M       Mdn   
model                                                                    
CNN       0.019089  0.010420  8.841193  106.753009  0.056249  0.041999   
CNN_LSTM  0.014212  0.006736  9.383533  103.807379  0.014125  0.007375   
LSTM      0.014201  0.007074  9.195510  109.425004  0.016194  0.010481   

data                            OHLCV_FOB                                  
               Skew        Kurt         M       Mdn      Skew        Kurt  
model                                                                      
CNN        3.115163   14.119886  0.018430  0.010229  9.246033  115.284693  
CNN_LSTM  10.118810  134.723838  0.010425  0.006444  5.113143   34.866629  
LSTM      11.062979  159.354731  0.015216  0.007383  8.023178   79.760156

M   Mdn      Skew      Kurt
data      model                                       
OHLCV_FOB CNN       0.507161  0.52 -0.086947 -1.136323
FOB       LSTM      0.505747  0.50 -0.043664 -1.302634
OHLCV_FOB LSTM      0.506300  0.50 -0.049052 -1.309488
FOB       CNN       0.490683  0.49  0.029466 -1.040280
OHLCV     CNN       0.487807  0.49  0.004201 -0.634267
OHLCV_FOB CNN_LSTM  0.486848  0.49  0.011591 -1.418633
FOB       CNN_LSTM  0.475951  0.47  0.061997 -1.347348
OHLCV     CNN_LSTM  0.472982  0.44  0.113541 -1.365599
          LSTM      0.445896  0.41  0.210963 -1.236670

data           FOB                               OHLCV                  \
                 M   Mdn      Skew      Kurt         M   Mdn      Skew   
model                                                                    
CNN       0.490683  0.49  0.029466 -1.040280  0.487807  0.49  0.004201   
CNN_LSTM  0.475951  0.47  0.061997 -1.347348  0.472982  0.44  0.113541   
LSTM      0.505747  0.50 -0.043664 -1.302634  0.445896  0.41  0.210963   

data               OHLCV_FOB                            
              Kurt         M   Mdn      Skew      Kurt  
model                                                   
CNN      -0.634267  0.507161  0.52 -0.086947 -1.136323  
CNN_LSTM -1.365599  0.486848  0.49  0.011591 -1.418633  
LSTM     -1.236670  0.506300  0.50 -0.049052 -1.309488

In [29]:
df['comb'] = df[['data','model']].apply(lambda row: '_'.join(row.values), axis=1)
comb = list(df['comb'].unique()) 
display(comb)

['FOB_LSTM',
 'OHLCV_LSTM',
 'OHLCV_CNN_LSTM',
 'FOB_CNN',
 'OHLCV_FOB_CNN_LSTM',
 'OHLCV_FOB_CNN',
 'FOB_CNN_LSTM',
 'OHLCV_CNN',
 'OHLCV_FOB_LSTM']

In [33]:
for m in ['mse','mae','accuracy']:
    groups = [df.loc[df['comb'] == c, m] for c in comb]
    stat, p_value = stats.kruskal(*groups)
    display(m, p_value)
    results_posthoc = sp.posthoc_dunn(groups, p_adjust='bonferroni')

    results_posthoc.index = comb
    results_posthoc.columns = comb
    #results_posthoc = results_posthoc.applymap(lambda x: '{:.4f}'.format(x))

    display('p_values', results_posthoc)
    
    medians = [np.median(g) for g in groups]

    median_diffs = pd.DataFrame(
        [[(m1 - m2) for m2 in medians] for m1 in medians],
        columns=comb,
        index=comb
    )

    #median_diffs = median_diffs.applymap(lambda x: '{:.1e}'.format(x))
    #display('median diff',median_diffs)
    
    df_stared = pd.DataFrame(
    np.vectorize(add_stars)(median_diffs.values, results_posthoc.values),
    columns=median_diffs.columns,
    index=median_diffs.index
    )
    
    for i in range(df_stared.shape[0]):
        for j in range(df_stared.shape[1]):
            if i >= j:
                df_stared.iat[i, j] = " "  # ou np.nan si tu préfères
    
    display(df_stared)
    df_stared.to_csv(f"median_diff_{m}.csv")

'mse'

0.0

'p_values'

,FOB_LSTM,OHLCV_LSTM,OHLCV_CNN_LSTM,FOB_CNN,OHLCV_FOB_CNN_LSTM,OHLCV_FOB_CNN,FOB_CNN_LSTM,OHLCV_CNN,OHLCV_FOB_LSTM
FOB_LSTM,1.000000e+00,4.640460e-226,5.098663e-06,0.000000e+00,2.706704e-16,6.796001e-266,9.302116e-02,0.0,7.945431e-03
OHLCV_LSTM,4.640460e-226,1.000000e+00,6.183054e-155,7.567653e-16,4.107477e-261,4.466713e-04,8.547324e-201,0.0,6.638933e-174
OHLCV_CNN_LSTM,5.098663e-06,6.183054e-155,1.000000e+00,1.276052e-258,3.695426e-36,1.427662e-191,2.566391e-12,0.0,1.000000e+00
FOB_CNN,0.000000e+00,7.567653e-16,1.276052e-258,1.000000e+00,0.000000e+00,5.546532e-03,3.984814e-299,0.0,1.731024e-282
OHLCV_FOB_CNN_LSTM,2.706704e-16,4.107477e-261,3.695426e-36,0.000000e+00,1.000000e+00,1.678396e-298,1.006567e-05,0.0,2.263829e-29
OHLCV_FOB_CNN,6.796001e-266,4.466713e-04,1.427662e-191,5.546532e-03,1.678396e-298,1.000000e+00,9.314240e-237,0.0,1.482971e-211
FOB_CNN_LSTM,9.302116e-02,8.547324e-201,2.566391e-12,3.984814e-299,1.006567e-05,9.314240e-237,1.000000e+00,0.0,2.829510e-08
OHLCV_CNN,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.0,0.000000e+00
OHLCV_FOB_LSTM,7.945431e-03,6.638933e-174,1.000000e+00,1.731024e-282,2.263829e-29,1.482971e-211,2.829510e-08,0.0,1.000000e+00


,FOB_LSTM,OHLCV_LSTM,OHLCV_CNN_LSTM,FOB_CNN,OHLCV_FOB_CNN_LSTM,OHLCV_FOB_CNN,FOB_CNN_LSTM,OHLCV_CNN,OHLCV_FOB_LSTM
FOB_LSTM,,-8.3e-05***,-4.6e-06***,-8.2e-05***,1.1e-05***,-7.6e-05***,5.8e-06,-2.6e-03***,-7.8e-06**
OHLCV_LSTM,,,7.8e-05***,5.7e-07***,9.4e-05***,7.4e-06***,8.9e-05***,-2.6e-03***,7.5e-05***
OHLCV_CNN_LSTM,,,,-7.8e-05***,1.6e-05***,-7.1e-05***,1.0e-05***,-2.6e-03***,-3.1e-06
FOB_CNN,,,,,9.3e-05***,6.8e-06**,8.8e-05***,-2.6e-03***,7.5e-05***
OHLCV_FOB_CNN_LSTM,,,,,,-8.7e-05***,-5.2e-06***,-2.7e-03***,-1.9e-05***
OHLCV_FOB_CNN,,,,,,,8.1e-05***,-2.6e-03***,6.8e-05***
FOB_CNN_LSTM,,,,,,,,-2.6e-03***,-1.4e-05***
OHLCV_CNN,,,,,,,,,2.6e-03***
OHLCV_FOB_LSTM,,,,,,,,,


...

'accuracy'

1.1325620646339174e-109

'p_values'

,FOB_LSTM,OHLCV_LSTM,OHLCV_CNN_LSTM,FOB_CNN,OHLCV_FOB_CNN_LSTM,OHLCV_FOB_CNN,FOB_CNN_LSTM,OHLCV_CNN,OHLCV_FOB_LSTM
FOB_LSTM,1.000000e+00,3.771679e-67,1.751316e-19,2.509825e-03,2.867924e-04,1.000000e+00,5.043369e-12,2.162257e-04,1.000000e+00
OHLCV_LSTM,3.771679e-67,1.000000e+00,1.373931e-13,3.453030e-39,2.728039e-20,1.084356e-67,5.398870e-12,1.062776e-36,2.689393e-67
OHLCV_CNN_LSTM,1.751316e-19,1.373931e-13,1.000000e+00,2.426518e-06,5.167686e-02,4.518775e-21,1.000000e+00,3.163428e-05,8.456391e-20
FOB_CNN,2.509825e-03,3.453030e-39,2.426518e-06,1.000000e+00,1.000000e+00,1.912424e-04,2.604542e-03,1.000000e+00,1.550861e-03
OHLCV_FOB_CNN_LSTM,2.867924e-04,2.728039e-20,5.167686e-02,1.000000e+00,1.000000e+00,2.677081e-05,7.804983e-01,1.000000e+00,1.839267e-04
OHLCV_FOB_CNN,1.000000e+00,1.084356e-67,4.518775e-21,1.912424e-04,2.677081e-05,1.000000e+00,1.929002e-13,1.334277e-05,1.000000e+00
FOB_CNN_LSTM,5.043369e-12,5.398870e-12,1.000000e+00,2.604542e-03,7.804983e-01,1.929002e-13,1.000000e+00,1.415214e-02,2.685410e-12
OHLCV_CNN,2.162257e-04,1.062776e-36,3.163428e-05,1.000000e+00,1.000000e+00,1.334277e-05,1.415214e-02,1.000000e+00,1.278513e-04
OHLCV_FOB_LSTM,1.000000e+00,2.689393e-67,8.456391e-20,1.550861e-03,1.839267e-04,1.000000e+00,2.685410e-12,1.278513e-04,1.000000e+00


,FOB_LSTM,OHLCV_LSTM,OHLCV_CNN_LSTM,FOB_CNN,OHLCV_FOB_CNN_LSTM,OHLCV_FOB_CNN,FOB_CNN_LSTM,OHLCV_CNN,OHLCV_FOB_LSTM
FOB_LSTM,,9.0e-02***,6.0e-02***,1.0e-02**,1.0e-02***,-2.0e-02,3.0e-02***,1.0e-02***,0.0e+00
OHLCV_LSTM,,,-3.0e-02***,-8.0e-02***,-8.0e-02***,-1.1e-01***,-6.0e-02***,-8.0e-02***,-9.0e-02***
OHLCV_CNN_LSTM,,,,-5.0e-02***,-5.0e-02,-8.0e-02***,-3.0e-02,-5.0e-02***,-6.0e-02***
FOB_CNN,,,,,0.0e+00,-3.0e-02***,2.0e-02**,0.0e+00,-1.0e-02**
OHLCV_FOB_CNN_LSTM,,,,,,-3.0e-02***,2.0e-02,0.0e+00,-1.0e-02***
OHLCV_FOB_CNN,,,,,,,5.0e-02***,3.0e-02***,2.0e-02
FOB_CNN_LSTM,,,,,,,,-2.0e-02*,-3.0e-02***
OHLCV_CNN,,,,,,,,,-1.0e-02***
OHLCV_FOB_LSTM,,,,,,,,,


# tmp grad

In [57]:
df = pd.read_parquet(os.path.join(metrics_path, "metrics_row.parquet.gzip"))
d = ['OHLCV','FOB']
m = ['CNN','LSTM']
df['data'] = df['data_mod'].apply(lambda x: '_'.join([i for i in x.split('_') if i in d]))
df['model'] = df['data_mod'].apply(lambda x: '_'.join([i for i in x.split('_') if i in m]))

def assigning(x,m):
    res = [np.polyfit(x['id_forecast'].values, x[m].values, 1)[0]] * len(x)
    return pd.Series(res, index=x.index)

for m in ['mse','mae','accuracy']:
    
    df[f'grad_{m}'] = df.groupby(['isin', 'data', 'model']).apply(lambda x: assigning(x,m)).reset_index(level=[0,1,2], drop=True)


df = df.groupby(['isin', 'data', 'model'], as_index=False)[['grad_mse','grad_mae','grad_accuracy']].last()
display(df)

for m in ['grad_mse']:
    desc = df.groupby(['data','model'])[m].describe()
    pivot = desc.copy()
    pivot['skew'] = df.groupby(['data','model'])[m].skew()
    pivot['kurt'] = df.groupby(['data','model'])[m].apply(pd.DataFrame.kurt)
    pivot = pivot[['mean', '50%', 'skew', 'kurt']]
    pivot.columns = ['M', 'Mdn', 'Skew', 'Kurt']
    display(pivot.sort_values('Mdn'))
    pivot[['M', 'Mdn']] = pivot[['M', 'Mdn']].applymap(lambda x: '{:.1e}'.format(x))
    if m == 'grad_accuracy':
        pivot[['Skew', 'Kurt']] = pivot[['Skew', 'Kurt']].applymap(lambda x: '{:.1f}'.format(x))
    else:
        pivot[['Skew', 'Kurt']] = pivot[['Skew', 'Kurt']].applymap(lambda x: '{:.0f}'.format(x))
        
    pivot = pivot.unstack(0).swaplevel(axis=1).sort_index(axis=1, level=0, sort_remaining=False)#.round(2)
    #pivot.to_csv(f'grad_tmp_{m}.csv')
    display(pivot)

,isin,data,model,grad_mse,grad_mae,grad_accuracy
0,FR0000045072,FOB,CNN,-3.611087e-07,-1.998776e-05,0.000204
1,FR0000045072,FOB,LSTM,-4.717528e-08,9.339637e-07,-0.000041
2,FR0000045072,OHLCV,CNN,7.773505e-06,4.488997e-05,0.000013
3,FR0000045072,OHLCV,CNN_LSTM,1.039968e-08,9.103848e-07,-0.000057
4,FR0000045072,OHLCV,LSTM,6.579526e-10,1.502721e-06,-0.000004
...,...,...,...,...,...,...
287,NL00150001Q9,OHLCV,CNN,3.092149e-06,2.166476e-05,-0.000359
288,NL00150001Q9,OHLCV,CNN_LSTM,7.471313e-07,2.443745e-05,-0.000263
289,NL00150001Q9,OHLCV,LSTM,7.556255e-07,2.463962e-05,-0.000255
290,NL00150001Q9,OHLCV_FOB,CNN,4.011965e-07,1.154850e-05,-0.000075


M           Mdn      Skew       Kurt
data      model                                                    
FOB       CNN_LSTM -1.964080e-05 -8.746571e-08 -4.572186  20.935092
OHLCV_FOB CNN_LSTM  7.551335e-08 -6.853725e-08  0.000000   0.000000
OHLCV     CNN_LSTM -1.025116e-05  8.804255e-09 -5.973573  35.784381
FOB       LSTM     -8.631081e-06  2.218413e-08 -5.487159  32.882949
          CNN      -1.026932e-05  5.312958e-08 -5.633527  33.375035
OHLCV_FOB LSTM     -7.100562e-06  6.492963e-08 -4.634511  27.812389
OHLCV     LSTM     -1.029785e-05  2.287148e-07 -6.224936  38.829919
OHLCV_FOB CNN      -1.227722e-05  3.933760e-07 -5.613901  31.675927
OHLCV     CNN      -8.259368e-07  3.474861e-06 -5.062168  29.754314

data           FOB                         OHLCV                    OHLCV_FOB  \
                 M       Mdn Skew Kurt         M      Mdn Skew Kurt         M   
model                                                                           
CNN       -1.0e-05   5.3e-08   -6   33  -8.3e-07  3.5e-06   -5   30  -1.2e-05   
CNN_LSTM  -2.0e-05  -8.7e-08   -5   21  -1.0e-05  8.8e-09   -6   36   7.6e-08   
LSTM      -8.6e-06   2.2e-08   -5   33  -1.0e-05  2.3e-07   -6   39  -7.1e-06   

data                          
               Mdn Skew Kurt  
model                         
CNN        3.9e-07   -6   32  
CNN_LSTM  -6.9e-08    0    0  
LSTM       6.5e-08   -5   28

In [54]:
df['comb'] = df[['data','model']].apply(lambda row: '_'.join(row.values), axis=1)
comb = list(df['comb'].unique()) 
display(comb)

for m in ['grad_mse']:
    groups = [df.loc[df['comb'] == c, m] for c in comb]
    stat, p_value = stats.kruskal(*groups)
    display(m, p_value)
    results_posthoc = sp.posthoc_dunn(groups, p_adjust='bonferroni')

    results_posthoc.index = comb
    results_posthoc.columns = comb
    #results_posthoc = results_posthoc.applymap(lambda x: '{:.4f}'.format(x))

    display('p_values', results_posthoc)
    
    medians = [np.median(g) for g in groups]

    median_diffs = pd.DataFrame(
        [[(m1 - m2) for m2 in medians] for m1 in medians],
        columns=comb,
        index=comb
    )

    #median_diffs = median_diffs.applymap(lambda x: '{:.1e}'.format(x))
    #display('median diff',median_diffs)
    
    df_stared = pd.DataFrame(
    np.vectorize(add_stars)(median_diffs.values, results_posthoc.values),
    columns=median_diffs.columns,
    index=median_diffs.index
    )
    
    for i in range(df_stared.shape[0]):
        for j in range(df_stared.shape[1]):
            if i >= j:
                df_stared.iat[i, j] = " "  # ou np.nan si tu préfères
    
    display(df_stared)
    df_stared.to_csv(f'grad_tmp_median_diff_{m}.csv')

['FOB_CNN',
 'FOB_LSTM',
 'OHLCV_CNN',
 'OHLCV_CNN_LSTM',
 'OHLCV_LSTM',
 'OHLCV_FOB_CNN',
 'OHLCV_FOB_CNN_LSTM',
 'OHLCV_FOB_LSTM',
 'FOB_CNN_LSTM']

'grad_mse'

0.0001831819466145109

'p_values'

,FOB_CNN,FOB_LSTM,OHLCV_CNN,OHLCV_CNN_LSTM,OHLCV_LSTM,OHLCV_FOB_CNN,OHLCV_FOB_CNN_LSTM,OHLCV_FOB_LSTM,FOB_CNN_LSTM
FOB_CNN,1.000000,1.000000,0.016769,1.000000,1.000000,1.0,1.000000,1.000000,1.000000
FOB_LSTM,1.000000,1.000000,0.002046,1.000000,1.000000,1.0,1.000000,1.000000,1.000000
OHLCV_CNN,0.016769,0.002046,1.000000,0.001851,0.191604,1.0,0.003848,0.016161,0.004051
OHLCV_CNN_LSTM,1.000000,1.000000,0.001851,1.000000,1.000000,1.0,1.000000,1.000000,1.000000
OHLCV_LSTM,1.000000,1.000000,0.191604,1.000000,1.000000,1.0,1.000000,1.000000,1.000000
OHLCV_FOB_CNN,1.000000,1.000000,1.000000,1.000000,1.000000,1.0,1.000000,1.000000,1.000000
OHLCV_FOB_CNN_LSTM,1.000000,1.000000,0.003848,1.000000,1.000000,1.0,1.000000,1.000000,1.000000
OHLCV_FOB_LSTM,1.000000,1.000000,0.016161,1.000000,1.000000,1.0,1.000000,1.000000,1.000000
FOB_CNN_LSTM,1.000000,1.000000,0.004051,1.000000,1.000000,1.0,1.000000,1.000000,1.000000


,FOB_CNN,FOB_LSTM,OHLCV_CNN,OHLCV_CNN_LSTM,OHLCV_LSTM,OHLCV_FOB_CNN,OHLCV_FOB_CNN_LSTM,OHLCV_FOB_LSTM,FOB_CNN_LSTM
FOB_CNN,,3.1e-08,-3.4e-06*,4.4e-08,-1.8e-07,-3.4e-07,1.2e-07,-1.2e-08,1.4e-07
FOB_LSTM,,,-3.5e-06**,1.3e-08,-2.1e-07,-3.7e-07,9.1e-08,-4.3e-08,1.1e-07
OHLCV_CNN,,,,3.5e-06**,3.2e-06,3.1e-06,3.5e-06**,3.4e-06*,3.6e-06**
OHLCV_CNN_LSTM,,,,,-2.2e-07,-3.8e-07,7.7e-08,-5.6e-08,9.6e-08
OHLCV_LSTM,,,,,,-1.6e-07,3.0e-07,1.6e-07,3.2e-07
OHLCV_FOB_CNN,,,,,,,4.6e-07,3.3e-07,4.8e-07
OHLCV_FOB_CNN_LSTM,,,,,,,,-1.3e-07,1.9e-08
OHLCV_FOB_LSTM,,,,,,,,,1.5e-07
FOB_CNN_LSTM,,,,,,,,,


# targ grad

In [58]:
df = pd.read_parquet(os.path.join(metrics_path,"metrics_col.parquet.gzip")).reset_index(drop=True)
d = ['OHLCV','FOB']
m = ['CNN','LSTM']
df['data'] = df['data_mod'].apply(lambda x: '_'.join([i for i in x.split('_') if i in d]))
df['model'] = df['data_mod'].apply(lambda x: '_'.join([i for i in x.split('_') if i in m]))
display(df)

def assigning(x,m):
    res = [np.polyfit(x['target'].values, x[m].values, 1)[0]] * len(x)
    return pd.Series(res, index=x.index)

for m in ['mse','mae','accuracy']:        
    df[f'grad_{m}'] = df.groupby(['isin', 'data', 'model']).apply(lambda x: assigning(x,m)).reset_index(level=[0,1,2], drop=True)
    
df = df.groupby(['isin', 'data', 'model'], as_index=False)[['grad_mse','grad_mae','grad_accuracy']].last()

for m in ['grad_mse']:
    desc = df.groupby(['data','model'])[m].describe()
    display(df.groupby(['data','model'])[m].describe())
    pivot = desc.copy()
    pivot['skew'] = df.groupby(['data','model'])[m].skew()
    pivot['kurt'] = df.groupby(['data','model'])[m].apply(pd.DataFrame.kurt)
    pivot = pivot[['mean', '50%', 'skew', 'kurt']]
    pivot.columns = ['M', 'Mdn', 'Skew', 'Kurt']
    display(pivot.sort_values('Mdn'))
    pivot[['M', 'Mdn']] = pivot[['M', 'Mdn']].applymap(lambda x: '{:.1e}'.format(x))
    pivot[['Skew', 'Kurt']] = pivot[['Skew', 'Kurt']].applymap(lambda x: '{:.0f}'.format(x))
    pivot = pivot.unstack(0).swaplevel(axis=1).sort_index(axis=1, level=0, sort_remaining=False)#.round(2)
    #pivot.to_csv(f'grad_targ_{m}.csv')
    display(pivot)



,mse,isin,data_mod,target,accuracy,mae,data,model
0,2.729047e-09,FR0000130577,FOB_LSTM,1,0.000000,0.000052,FOB,LSTM
1,5.336917e-06,FR0000130577,FOB_LSTM,2,0.469849,0.001081,FOB,LSTM
2,1.064642e-05,FR0000130577,FOB_LSTM,3,0.494975,0.001645,FOB,LSTM
3,1.445356e-05,FR0000130577,FOB_LSTM,4,0.484925,0.002036,FOB,LSTM
4,1.746610e-05,FR0000130577,FOB_LSTM,5,0.527638,0.002356,FOB,LSTM
...,...,...,...,...,...,...,...,...
29195,6.557299e-04,FR0000120172,FOB_CNN_LSTM,96,0.236181,0.018998,FOB,CNN_LSTM
29196,6.618448e-04,FR0000120172,FOB_CNN_LSTM,97,0.238693,0.019134,FOB,CNN_LSTM
29197,6.684086e-04,FR0000120172,FOB_CNN_LSTM,98,0.233668,0.019291,FOB,CNN_LSTM
29198,6.766942e-04,FR0000120172,FOB_CNN_LSTM,99,0.231156,0.019499,FOB,CNN_LSTM


count      mean       std           min           25%  \
data      model                                                             
FOB       CNN        36.0  0.000040  0.000148  7.267294e-07  2.536382e-06   
          CNN_LSTM   21.0  0.000048  0.000184  4.534868e-07  7.323972e-07   
          LSTM       37.0  0.000035  0.000142  3.922000e-07  1.153085e-06   
OHLCV     CNN        37.0  0.000166  0.000427 -6.197025e-06  7.099411e-06   
          CNN_LSTM   36.0  0.000031  0.000137  3.940968e-07  1.349614e-06   
          LSTM       39.0  0.000034  0.000136  3.875982e-07  2.236279e-06   
OHLCV_FOB CNN        32.0  0.000042  0.000176  5.900696e-07  1.199171e-06   
          CNN_LSTM   18.0  0.000010  0.000020  3.368976e-07  1.186056e-06   
          LSTM       36.0  0.000039  0.000150  3.602741e-07  1.417562e-06   

                         50%       75%       max  
data      model                                   
FOB       CNN       0.000004  0.000013  0.000879  
          CNN_LSTM  0.000002  0.000004  0.000846  
          LSTM      0.000002  0.000005  0.000847  
OHLCV     CNN       0.000036  0.000080  0.002293  
          CNN_LSTM  0.000002  0.000005  0.000823  
          LSTM      0.000005  0.000011  0.000853  
OHLCV_FOB CNN       0.000004  0.000009  0.000997  
          CNN_LSTM  0.000002  0.000003  0.000073  
          LSTM      0.000002  0.000007  0.000854

M       Mdn      Skew       Kurt
data      model                                            
OHLCV_FOB CNN_LSTM  0.000010  0.000002  2.607813   6.494377
FOB       CNN_LSTM  0.000048  0.000002  4.516084  20.555609
OHLCV_FOB LSTM      0.000039  0.000002  5.032658  26.584042
FOB       LSTM      0.000035  0.000002  5.506276  31.588464
OHLCV     CNN_LSTM  0.000031  0.000002  5.855931  34.767549
OHLCV_FOB CNN       0.000042  0.000004  5.493545  30.650932
FOB       CNN       0.000040  0.000004  5.521320  31.620754
OHLCV     LSTM      0.000034  0.000005  6.051285  37.277022
          CNN       0.000166  0.000036  4.073537  18.153554

data          FOB                       OHLCV                    OHLCV_FOB  \
                M      Mdn Skew Kurt        M      Mdn Skew Kurt         M   
model                                                                        
CNN       4.0e-05  4.4e-06    6   32  1.7e-04  3.6e-05    4   18   4.2e-05   
CNN_LSTM  4.8e-05  2.2e-06    5   21  3.1e-05  2.5e-06    6   35   9.5e-06   
LSTM      3.5e-05  2.2e-06    6   32  3.4e-05  4.9e-06    6   37   3.9e-05   

data                         
              Mdn Skew Kurt  
model                        
CNN       3.6e-06    5   31  
CNN_LSTM  2.0e-06    3    6  
LSTM      2.2e-06    5   27

In [56]:
df['comb'] = df[['data','model']].apply(lambda row: '_'.join(row.values), axis=1)
comb = list(df['comb'].unique()) 
display(comb)

for m in ['grad_mse']:
    groups = [df.loc[df['comb'] == c, m] for c in comb]
    stat, p_value = stats.kruskal(*groups)
    display(m, p_value)
    results_posthoc = sp.posthoc_dunn(groups, p_adjust='bonferroni')

    results_posthoc.index = comb
    results_posthoc.columns = comb
    #results_posthoc = results_posthoc.applymap(lambda x: '{:.4f}'.format(x))

    display('p_values', results_posthoc)
    
    medians = [np.median(g) for g in groups]

    median_diffs = pd.DataFrame(
        [[(m1 - m2) for m2 in medians] for m1 in medians],
        columns=comb,
        index=comb
    )

    #median_diffs = median_diffs.applymap(lambda x: '{:.1e}'.format(x))
    #display('median diff',median_diffs)
    
    df_stared = pd.DataFrame(
    np.vectorize(add_stars)(median_diffs.values, results_posthoc.values),
    columns=median_diffs.columns,
    index=median_diffs.index
    )
    
    for i in range(df_stared.shape[0]):
        for j in range(df_stared.shape[1]):
            if i >= j:
                df_stared.iat[i, j] = " "  # ou np.nan si tu préfères
    
    display(df_stared)
    df_stared.to_csv(f'grad_targ_median_diff_{m}.csv')

['FOB_CNN',
 'FOB_LSTM',
 'OHLCV_CNN',
 'OHLCV_CNN_LSTM',
 'OHLCV_LSTM',
 'OHLCV_FOB_CNN',
 'OHLCV_FOB_CNN_LSTM',
 'OHLCV_FOB_LSTM',
 'FOB_CNN_LSTM']

'grad_mse'

4.8701450789253866e-05

'p_values'

,FOB_CNN,FOB_LSTM,OHLCV_CNN,OHLCV_CNN_LSTM,OHLCV_LSTM,OHLCV_FOB_CNN,OHLCV_FOB_CNN_LSTM,OHLCV_FOB_LSTM,FOB_CNN_LSTM
FOB_CNN,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.798666,1.000000,1.000000
FOB_LSTM,1.000000,1.000000,0.000779,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
OHLCV_CNN,1.000000,0.000779,1.000000,0.002512,0.681425,0.056615,0.002151,0.004145,0.004028
OHLCV_CNN_LSTM,1.000000,1.000000,0.002512,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
OHLCV_LSTM,1.000000,1.000000,0.681425,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
OHLCV_FOB_CNN,1.000000,1.000000,0.056615,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
OHLCV_FOB_CNN_LSTM,0.798666,1.000000,0.002151,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
OHLCV_FOB_LSTM,1.000000,1.000000,0.004145,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
FOB_CNN_LSTM,1.000000,1.000000,0.004028,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


,FOB_CNN,FOB_LSTM,OHLCV_CNN,OHLCV_CNN_LSTM,OHLCV_LSTM,OHLCV_FOB_CNN,OHLCV_FOB_CNN_LSTM,OHLCV_FOB_LSTM,FOB_CNN_LSTM
FOB_CNN,,2.2e-06,-3.1e-05,1.9e-06,-5.3e-07,7.8e-07,2.4e-06,2.2e-06,2.2e-06
FOB_LSTM,,,-3.3e-05***,-2.8e-07,-2.7e-06,-1.4e-06,2.1e-07,1.4e-10,9.5e-09
OHLCV_CNN,,,,3.3e-05**,3.1e-05,3.2e-05,3.4e-05**,3.3e-05**,3.3e-05**
OHLCV_CNN_LSTM,,,,,-2.4e-06,-1.1e-06,4.9e-07,2.8e-07,2.9e-07
OHLCV_LSTM,,,,,,1.3e-06,2.9e-06,2.7e-06,2.7e-06
OHLCV_FOB_CNN,,,,,,,1.6e-06,1.4e-06,1.4e-06
OHLCV_FOB_CNN_LSTM,,,,,,,,-2.1e-07,-2.0e-07
OHLCV_FOB_LSTM,,,,,,,,,9.3e-09
FOB_CNN_LSTM,,,,,,,,,


In [7]:
df = pd.read_parquet(os.path.join(metrics_path,"metrics_col.parquet.gzip")).reset_index(drop=True)
d = ['OHLCV','FOB']
m = ['CNN','LSTM']
df['data'] = df['data_mod'].apply(lambda x: '_'.join([i for i in x.split('_') if i in d]))
df['model'] = df['data_mod'].apply(lambda x: '_'.join([i for i in x.split('_') if i in m]))
display(df)

for m in ['mse','mae','accuracy']:
    desc = df.groupby(['data','model'])[m].describe()
    pivot = desc.copy()
    pivot['skew'] = df.groupby(['data','model'])[m].skew()
    pivot['kurt'] = df.groupby(['data','model'])[m].apply(pd.DataFrame.kurt)
    pivot = pivot[['mean', '50%', 'skew', 'kurt']]
    pivot.columns = ['M', 'Mdn', 'Skew', 'Kurt']
    pivot = pivot.unstack(0).swaplevel(axis=1).sort_index(axis=1, level=0, sort_remaining=False)#.round(2)
    display(pivot)

,mse,isin,data_mod,target,accuracy,mae,data,model
0,2.729047e-09,FR0000130577,FOB_LSTM,1,0.000000,0.000052,FOB,LSTM
1,5.336917e-06,FR0000130577,FOB_LSTM,2,0.469849,0.001081,FOB,LSTM
2,1.064642e-05,FR0000130577,FOB_LSTM,3,0.494975,0.001645,FOB,LSTM
3,1.445356e-05,FR0000130577,FOB_LSTM,4,0.484925,0.002036,FOB,LSTM
4,1.746610e-05,FR0000130577,FOB_LSTM,5,0.527638,0.002356,FOB,LSTM
...,...,...,...,...,...,...,...,...
29195,6.557299e-04,FR0000120172,FOB_CNN_LSTM,96,0.236181,0.018998,FOB,CNN_LSTM
29196,6.618448e-04,FR0000120172,FOB_CNN_LSTM,97,0.238693,0.019134,FOB,CNN_LSTM
29197,6.684086e-04,FR0000120172,FOB_CNN_LSTM,98,0.233668,0.019291,FOB,CNN_LSTM
29198,6.766942e-04,FR0000120172,FOB_CNN_LSTM,99,0.231156,0.019499,FOB,CNN_LSTM


data           FOB                                    OHLCV            \
                 M       Mdn      Skew       Kurt         M       Mdn   
model                                                                   
CNN       0.002125  0.000212  6.934817  51.201358  0.008870  0.001407   
CNN_LSTM  0.002306  0.000098  5.659715  32.529546  0.001490  0.000111   
LSTM      0.001745  0.000111  6.858748  50.987701  0.001548  0.000172   

data                          OHLCV_FOB                                 
              Skew       Kurt         M       Mdn      Skew       Kurt  
model                                                                   
CNN       5.762900  42.783766  0.002116  0.000201  7.119956  53.710125  
CNN_LSTM  7.530564  59.594755  0.000442  0.000094  3.649699  14.328583  
LSTM      7.777085  63.494087  0.002089  0.000114  6.335761  44.229623

data           FOB                                    OHLCV            \
                 M       Mdn      Skew       Kurt         M       Mdn   
model                                                                   
CNN       0.019111  0.011526  3.771655  17.509835  0.056287  0.033554   
CNN_LSTM  0.014234  0.007527  4.600615  23.025139  0.014141  0.008189   
LSTM      0.014224  0.008032  4.398759  21.186600  0.016206  0.010395   

data                          OHLCV_FOB                                 
              Skew       Kurt         M       Mdn      Skew       Kurt  
model                                                                   
CNN       2.857977  10.743888  0.018450  0.011147  5.094428  31.791920  
CNN_LSTM  5.087119  30.288154  0.010428  0.007410  2.380414   5.379903  
LSTM      4.408735  25.231126  0.015241  0.008347  4.809300  26.438081

data           FOB                                   OHLCV            \
                 M       Mdn      Skew      Kurt         M       Mdn   
model                                                                  
CNN       0.490679  0.489950 -0.323941  2.804702  0.487808  0.479899   
CNN_LSTM  0.475949  0.469849 -0.466170  1.996297  0.472967  0.459799   
LSTM      0.505727  0.492462 -0.262764  2.303738  0.445899  0.444444   

data                         OHLCV_FOB                                
              Skew      Kurt         M       Mdn      Skew      Kurt  
model                                                                 
CNN      -0.213471  1.081823  0.507149  0.502513 -0.301312  1.379792  
CNN_LSTM -0.052424  1.098338  0.486859  0.482412 -0.360743  1.865466  
LSTM     -0.219547  1.519705  0.506290  0.500000 -0.278453  2.127264

In [22]:
import pandas as pd
import numpy as np

# Exemple de DataFrames
df_diff = pd.DataFrame({
    'A': [1.2e-4, -5.6e-3, 8.9e-2],
    'B': [2.1e3, -1.3e2, 0.0]
})

df_pval = pd.DataFrame({
    'A': [0.04, 0.2, 0.0005],
    'B': [0.07, 0.008, 0.5]
})

# Fonction pour format scientifique + astérisques
def add_stars(diff, p):
    diff_str = f"{diff:.2e}"  # écriture scientifique, 2 décimales
    if p < 0.001:
        stars = '***'
    elif p < 0.01:
        stars = '**'
    elif p < 0.05:
        stars = '*'
    else:
        stars = ''
    return f"{diff_str}{stars}"

# Application vectorisée
df_stared = pd.DataFrame(
    np.vectorize(add_stars)(df_diff.values, df_pval.values),
    columns=df_diff.columns,
    index=df_diff.index
)

print(df_stared)


             A            B
0    1.20e-04*     2.10e+03
1    -5.60e-03  -1.30e+02**
2  8.90e-02***     0.00e+00
